# PLACES Health Data - Data Cleaning (Data Wrangler Notebook)

**Group 3:** Salam Odeh, Qossay Shtaiwi, Ali Odah

**Dataset:** PLACES: Local Data for Better Health, Census Tract Data (2024 release) — CDC / U.S. Department of Health & Human Services

**Role:** Data Wrangler (Ali Odah)

**Goal of this notebook:** take the raw PLACES export (one row per census tract *per health measure*) and turn it into a single clean, modeling-ready table (one row *per census tract*, one column *per health measure*), so the Analyzer and Modeler can build directly on top of it.

**Target variable for modeling:** `Diabetes` (crude prevalence, % of adults) — chosen because it's a well-understood health outcome with strong, interpretable predictors already present in this dataset (obesity, physical inactivity, access to care, etc.).

**Tasks:**
1. Load the data and do an initial inspection
2. Understand the data's shape: one row = one tract + one measure
3. cleans and validates the data
4. Pivot the data from long format to wide format (one row per tract)
5. handles missing values
6. remove very small populations
7. saves the final dataset with a data dictionary.


In [1]:
# Import the required libraries
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Define the file path
fpath = "/content/drive/MyDrive/export.csv"

# Keep geographic identifiers as text
id_columns = {
    "CountyFIPS": "string",
    "LocationName": "string",
    "LocationID": "string"
}

# Load the dataset
df = pd.read_csv(
    fpath,
    dtype=id_columns,
    low_memory=False
)


In [4]:
# Inspect the dataset
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())
df.info()

print("\nUnique categories:", df["Category"].nunique())
print("Unique measures:", df["Measure"].nunique())
print("\nData value types:")
print(df["Data_Value_Type"].value_counts())


Number of rows: 3047284
Number of columns: 24


,Year,StateAbbr,StateDesc,CountyName,CountyFIPS,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,Data_Value_Footnote_Symbol,Data_Value_Footnote,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,Geolocation,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text
0,2023,AL,Alabama,Jefferson,01073,01073010900,BRFSS,Disability,Any disability among adults,%,Crude prevalence,44.9,NaN,NaN,40.7,49.0,"4,719","3,410",POINT (-86.7705771 33.5795996),01073010900,DISABLT,DISABILITY,CrdPrv,Any Disability
1,2023,AL,Alabama,Jefferson,01073,01073011207,BRFSS,Health Outcomes,Depression among adults,%,Crude prevalence,22.7,NaN,NaN,19.9,25.9,"5,103","3,684",POINT (-86.6742325 33.6638358),01073011207,HLTHOUT,DEPRESSION,CrdPrv,Depression
2,2023,AL,Alabama,Lauderdale,01077,01077010100,BRFSS,Disability,Any disability among adults,%,Crude prevalence,40.3,NaN,NaN,34.0,46.2,"2,278","2,093",POINT (-87.6598801 34.7958382),01077010100,DISABLT,DISABILITY,CrdPrv,Any Disability
3,2023,AL,Alabama,Lee,01081,01081040202,BRFSS,Health Outcomes,Arthritis among adults,%,Crude prevalence,23.5,NaN,NaN,20.7,26.2,"3,084","2,459",POINT (-85.4521355 32.6113816),01081040202,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis
4,2023,AL,Alabama,Lee,01081,01081040300,BRFSS,Health Outcomes,Obesity among adults,%,Crude prevalence,35.5,NaN,NaN,29.2,42.2,"2,576","2,188",POINT (-85.4668576 32.6016654),01081040300,HLTHOUT,OBESITY,CrdPrv,Obesity


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3047284 entries, 0 to 3047283
Data columns (total 24 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Year                        int64  
 1   StateAbbr                   object 
 2   StateDesc                   object 
 3   CountyName                  object 
 4   CountyFIPS                  string 
 5   LocationName                string 
 6   DataSource                  object 
 7   Category                    object 
 8   Measure                     object 
 9   Data_Value_Unit             object 
 10  Data_Value_Type             object 
 11  Data_Value                  float64
 12  Data_Value_Footnote_Symbol  float64
 13  Data_Value_Footnote         float64
 14  Low_Confidence_Limit        float64
 15  High_Confidence_Limit       float64
 16  TotalPopulation             object 
 17  TotalPop18plus              object 
 18  Geolocation                 object 
 19  LocationID           

**Data Loading and Initial Inspection**

The dataset was loaded and inspected to understand its size, data types, categories, measures, and available value types. Geographic identifiers were kept as text to preserve leading zeros.


In [5]:
# Keep only crude-prevalence observations
df = df[
    df["Data_Value_Type"] == "Crude prevalence"
].reset_index(drop=True)

# Check whether any (tract, measure) pair appears in more than one year
# before deciding whether a year filter is needed
duplicate_tract_measure = df.duplicated(
    subset=["LocationID", "Short_Question_Text"]
).sum()

print("Years present:", sorted(df["Year"].unique()))
print("Duplicate (tract, measure) pairs across years:", duplicate_tract_measure)
print("Rows after filtering:", df.shape[0])
print(df["Data_Value_Type"].value_counts())


Years present: [np.int64(2022), np.int64(2023)]
Duplicate (tract, measure) pairs across years: 0
Rows after filtering: 3047284
Data_Value_Type
Crude prevalence    3047284
Name: count, dtype: int64


**Observation and Year Selection**

Only crude-prevalence observations were retained. The data actually spans two survey years (2022 and 2023), since PLACES reports several measures on a rotating two-year BRFSS cycle. We checked whether this creates any conflict for our pivot: no `(tract, measure)` pair appears in more than one year, so every measure has exactly one value per tract regardless of year. Filtering to a single "latest" year would therefore not resolve any real conflict — it would only *drop* the 5 measures that happen to be reported on the off-year (including `Dental Visit`, `Mammography`, `Short Sleep Duration`, `Colorectal Cancer Screening`, `All Teeth Lost`), shrinking the dataset from 40 usable measures to 35. So we keep all years here and let the pivot naturally combine them.


In [6]:
# Check missing values
missing_values = df.isna().sum()
missing_percentages = (
    missing_values / len(df) * 100
).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentages
})

display(
    missing_summary[
        missing_summary["Missing Values"] > 0
    ]
)

# Remove completely empty columns
empty_columns = missing_summary[
    missing_summary["Missing Percentage"] == 100
].index.tolist()

df.drop(
    columns=empty_columns,
    inplace=True
)

print("Removed empty columns:", empty_columns)
print("Remaining missing values:", df.isna().sum().sum())


,Missing Values,Missing Percentage
Data_Value_Footnote_Symbol,3047284,100.0
Data_Value_Footnote,3047284,100.0


Removed empty columns: ['Data_Value_Footnote_Symbol', 'Data_Value_Footnote']
Remaining missing values: 0


In [7]:
# Check duplicate rows and observations
exact_duplicates = df.duplicated().sum()

observation_key = [
    "Year",
    "LocationID",
    "Short_Question_Text"
]

observation_duplicates = df.duplicated(
    subset=observation_key
).sum()

print("Exact duplicate rows:", exact_duplicates)
print("Duplicated observations:", observation_duplicates)

# Remove duplicates if any are found
df.drop_duplicates(inplace=True)

df.drop_duplicates(
    subset=observation_key,
    inplace=True
)

df.reset_index(drop=True, inplace=True)


Exact duplicate rows: 0
Duplicated observations: 0


**Missing Values and Duplicate Records**

Completely empty columns were removed because they contained no useful information. The data was also checked for exact duplicate rows and repeated health observations.


In [8]:
# Convert population columns to numeric
population_columns = [
    "TotalPopulation",
    "TotalPop18plus"
]

for column in population_columns:
    df[column] = pd.to_numeric(
        df[column]
        .astype("string")
        .str.replace(",", "", regex=False)
    )

display(
    df[
        [
            "TotalPopulation",
            "TotalPop18plus"
        ]
    ].head()
)


,TotalPopulation,TotalPop18plus
0,4719,3410
1,5103,3684
2,2278,2093
3,3084,2459
4,2576,2188


**Data Type Conversion**

Population columns were converted from text to numeric values so they can be used for filtering and analysis.


In [9]:
# Display full descriptions with their shorter identifiers
display(
    df[
        ["StateDesc", "StateAbbr"]
    ].drop_duplicates().head()
)

display(
    df[
        ["Category", "CategoryID"]
    ].drop_duplicates()
)

display(
    df[
        ["Measure", "MeasureId", "Short_Question_Text"]
    ].drop_duplicates().head()
)

# Remove redundant descriptions and constant metadata
df.drop(
    columns=[
        "CategoryID",
        "Measure",
        "DataSource",
        "Data_Value_Unit",
        "Data_Value_Type",
        "DataValueTypeID"
    ],
    inplace=True
)


,StateDesc,StateAbbr
0,Alabama,AL
13,Alaska,AK
22,Arizona,AZ
28,Arkansas,AR
44,California,CA


,Category,CategoryID
0,Disability,DISABLT
1,Health Outcomes,HLTHOUT
1332,Health Risk Behaviors,RISKBEH
1380,Health Status,HLTHSTAT
1385,Health-Related Social Needs,SOCLNEED
1388,Prevention,PREVENT


,Measure,MeasureId,Short_Question_Text
0,Any disability among adults,DISABILITY,Any Disability
1,Depression among adults,DEPRESSION,Depression
3,Arthritis among adults,ARTHRITIS,Arthritis
4,Obesity among adults,OBESITY,Obesity
5,Stroke among adults,STROKE,Stroke


**Removing Redundant Columns**

Long descriptions and constant metadata were removed to reduce the dataset size. `StateAbbr`, `LocationName`, and `Short_Question_Text` were retained because they are useful for the final wide dataset.


In [10]:
# Check health percentages
invalid_percentages = (
    (df["Data_Value"] < 0) |
    (df["Data_Value"] > 100)
).sum()

# Check confidence limits
invalid_confidence_limits = (
    (df["Data_Value"] < df["Low_Confidence_Limit"]) |
    (df["Data_Value"] > df["High_Confidence_Limit"])
).sum()

# Check population values
invalid_population = (
    (df["TotalPopulation"] < 0) |
    (df["TotalPop18plus"] < 0) |
    (df["TotalPop18plus"] > df["TotalPopulation"])
).sum()



print("Percentages outside 0-100:", invalid_percentages)
print("Values outside confidence limits:", invalid_confidence_limits)
print("Invalid population records:", invalid_population)


Percentages outside 0-100: 0
Values outside confidence limits: 0
Invalid population records: 0


In [11]:
# Validate geographic identifiers
invalid_county_fips = (
    df["CountyFIPS"].str.len() != 5
).sum()

invalid_location_ids = (
    df["LocationID"].str.len() != 11
).sum()

fips_mismatches = (
    df["CountyFIPS"] !=
    df["LocationID"].str[:5]
).sum()

print("Invalid CountyFIPS codes:", invalid_county_fips)
print("Invalid LocationID codes:", invalid_location_ids)
print("CountyFIPS and LocationID mismatches:", fips_mismatches)


Invalid CountyFIPS codes: 0
Invalid LocationID codes: 0
CountyFIPS and LocationID mismatches: 0


**Data Quality Validation**

Health percentages, confidence limits, population relationships, coordinates, and geographic identifiers were checked before reshaping the data.


In [12]:
# Columns that describe each census tract
reference_columns = [
    "LocationName",
    "StateAbbr",
    "StateDesc",
    "CountyName",
    "CountyFIPS",
    "TotalPopulation",
    "TotalPop18plus",
    "Geolocation"
]

# Convert health measures from rows to columns
df_wide = (
    df.pivot_table(
        index=reference_columns,
        columns="Short_Question_Text",
        values="Data_Value",
        aggfunc="first"
    )
    .reset_index()
)

df_wide.columns.name = None

print("Long dataset shape:", df.shape)
print("Wide dataset shape:", df_wide.shape)

display(df_wide.head())


Long dataset shape: (3047284, 16)
Wide dataset shape: (83522, 48)


,LocationName,StateAbbr,StateDesc,CountyName,CountyFIPS,TotalPopulation,TotalPop18plus,Geolocation,All Teeth Lost,Annual Checkup,Any Disability,Arthritis,Binge Drinking,COPD,Cancer (non-skin) or Melanoma,Cholesterol Screening,Cognitive Disability,Colorectal Cancer Screening,Coronary Heart Disease,Current Asthma,Current Cigarette Smoking,Dental Visit,Depression,Diabetes,Food Insecurity,Food Stamps,Frequent Mental Distress,Frequent Physical Distress,General Health,Health Insurance,Hearing Disability,High Blood Pressure,High Blood Pressure Medication,High Cholesterol,Housing Insecurity,Independent Living Disability,Lack of Social/Emotional Support,Loneliness,Mammography,Mobility Disability,Obesity,Physical Inactivity,Self-care Disability,Short Sleep Duration,Stroke,Transportation Barriers,Utility Services Threat,Vision Disability
0,01001020100,AL,Alabama,Autauga,01001,1775,1370,POINT (-86.4915648 32.4819731),13.4,80.1,35.7,30.6,14.8,8.2,8.5,85.5,16.8,65.0,7.2,10.2,15.6,57.3,26.6,13.3,17.0,11.6,17.9,14.7,22.5,9.7,8.6,41.4,77.8,41.9,12.3,9.6,23.5,33.0,78.0,16.3,39.4,27.8,4.4,37.9,3.8,8.9,8.1,5.5
1,01001020200,AL,Alabama,Autauga,01001,2055,1584,POINT (-86.4724678 32.475758),15.9,81.8,36.9,29.5,12.9,7.6,6.3,85.6,17.3,68.1,6.5,10.8,16.7,56.1,23.3,15.8,24.8,18.4,18.5,14.4,25.1,10.6,6.7,45.9,78.3,39.7,18.5,10.4,29.1,36.1,79.2,17.8,44.7,32.0,5.0,42.6,4.4,12.0,12.5,6.6
2,01001020300,AL,Alabama,Autauga,01001,3216,2485,POINT (-86.4597033 32.4740243),14.7,80.6,37.7,31.5,13.9,8.5,8.5,85.5,17.6,67.4,7.3,10.7,16.1,57.7,26.3,13.9,19.1,13.1,18.6,15.1,23.7,10.6,8.6,42.9,78.4,41.8,14.1,10.3,25.9,35.1,75.5,17.6,40.3,29.9,4.6,39.0,4.1,9.7,9.0,5.8
3,01001020400,AL,Alabama,Autauga,01001,4246,3344,POINT (-86.4448353 32.4710304),9.3,81.7,33.9,32.5,14.2,7.6,10.3,88.0,14.2,71.5,7.5,9.6,12.9,61.1,24.9,12.8,12.5,7.9,15.8,13.5,19.5,7.9,9.3,42.4,80.1,43.4,9.3,8.3,21.5,31.0,78.1,15.5,36.9,25.5,3.6,35.8,3.8,6.8,6.1,4.7
4,01001020501,AL,Alabama,Autauga,01001,4322,3369,POINT (-86.4225578 32.4478607),9.9,81.3,27.5,27.7,15.6,5.2,9.0,89.0,12.0,70.5,5.4,9.3,10.0,66.3,24.0,10.2,9.5,5.4,14.8,11.0,14.9,6.2,6.8,37.2,77.8,40.5,8.1,6.3,21.1,30.8,79.8,11.2,34.7,20.3,2.6,35.1,2.8,5.5,5.1,3.3


**Long-to-Wide Transformation**

Each census tract is now represented by one row. The values from `Short_Question_Text` became separate health-measure columns, while state, county, population, and geolocation fields were kept as reference information.


In [13]:
# Identify the health-measure columns
measure_columns = [
    column for column in df_wide.columns
    if column not in reference_columns
]

# Calculate missing percentages after the pivot
measure_missing = (
    df_wide[measure_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(
    measure_missing.to_frame(
        "Missing Percentage"
    )
)


,Missing Percentage
Transportation Barriers,27.660976
Housing Insecurity,27.660976
Loneliness,27.660976
Utility Services Threat,27.660976
Food Insecurity,27.660976
Lack of Social/Emotional Support,27.660976
Food Stamps,27.660976
Annual Checkup,5.635641
Stroke,5.635641
Cholesterol Screening,5.635641


**Missing Values After the Pivot**

Missing values were reviewed for each health measure. Rows missing the target (`Diabetes`) will be removed, while remaining missing feature values will be filled using the median of each measure.


In [14]:
# Drop rows missing the target variable
before = df_wide.shape[0]
df_wide = df_wide.dropna(subset=["Diabetes"]).copy()
after = df_wide.shape[0]
print(f"Dropped {before - after} tracts missing the target (Diabetes).")
print(f"Remaining shape: {df_wide.shape}")

# Fill remaining missing feature values with the median
median_values = df_wide[measure_columns].median()
df_wide[measure_columns] = (
    df_wide[measure_columns]
    .fillna(median_values)
)

print(
    "Remaining missing measure values:",
    df_wide[measure_columns].isna().sum().sum()
)


Dropped 4707 tracts missing the target (Diabetes).
Remaining shape: (78815, 48)
Remaining missing measure values: 0


**Missing Values**

Rows missing the `Diabetes` target were removed because the target should not be imputed for modeling. Remaining missing health-measure values were filled using the median of each measure.


In [15]:
# Check population distribution
display(df_wide["TotalPopulation"].describe())

# Remove very small-population tracts
small_population_threshold = 100
tiny_tracts = (df_wide["TotalPopulation"] < small_population_threshold).sum()
print(f"Tracts with population under 100: {tiny_tracts}")

df_wide = df_wide[
    df_wide["TotalPopulation"] >= small_population_threshold
].copy()
print(f"Shape after removing tiny-population tracts: {df_wide.shape}")


,TotalPopulation
count,78815.0
mean,3983.20982
std,1682.817058
min,54.0
25%,2781.0
50%,3795.0
75%,4965.0
max,37892.0


Tracts with population under 100: 31
Shape after removing tiny-population tracts: (78784, 48)


**Small-Population Filter**

Census tracts with fewer than 100 residents were removed because their prevalence estimates are statistically less reliable.


In [16]:
# Final validation
print("Final rows:", df_wide.shape[0])
print("Final columns:", df_wide.shape[1])

print(
    "Duplicated LocationName values:",
    df_wide["LocationName"].duplicated().sum()
)

print(
    "Remaining missing values:",
    df_wide.isna().sum().sum()
)

display(df_wide.head())


Final rows: 78784
Final columns: 48
Duplicated LocationName values: 0
Remaining missing values: 0


,LocationName,StateAbbr,StateDesc,CountyName,CountyFIPS,TotalPopulation,TotalPop18plus,Geolocation,All Teeth Lost,Annual Checkup,Any Disability,Arthritis,Binge Drinking,COPD,Cancer (non-skin) or Melanoma,Cholesterol Screening,Cognitive Disability,Colorectal Cancer Screening,Coronary Heart Disease,Current Asthma,Current Cigarette Smoking,Dental Visit,Depression,Diabetes,Food Insecurity,Food Stamps,Frequent Mental Distress,Frequent Physical Distress,General Health,Health Insurance,Hearing Disability,High Blood Pressure,High Blood Pressure Medication,High Cholesterol,Housing Insecurity,Independent Living Disability,Lack of Social/Emotional Support,Loneliness,Mammography,Mobility Disability,Obesity,Physical Inactivity,Self-care Disability,Short Sleep Duration,Stroke,Transportation Barriers,Utility Services Threat,Vision Disability
0,01001020100,AL,Alabama,Autauga,01001,1775,1370,POINT (-86.4915648 32.4819731),13.4,80.1,35.7,30.6,14.8,8.2,8.5,85.5,16.8,65.0,7.2,10.2,15.6,57.3,26.6,13.3,17.0,11.6,17.9,14.7,22.5,9.7,8.6,41.4,77.8,41.9,12.3,9.6,23.5,33.0,78.0,16.3,39.4,27.8,4.4,37.9,3.8,8.9,8.1,5.5
1,01001020200,AL,Alabama,Autauga,01001,2055,1584,POINT (-86.4724678 32.475758),15.9,81.8,36.9,29.5,12.9,7.6,6.3,85.6,17.3,68.1,6.5,10.8,16.7,56.1,23.3,15.8,24.8,18.4,18.5,14.4,25.1,10.6,6.7,45.9,78.3,39.7,18.5,10.4,29.1,36.1,79.2,17.8,44.7,32.0,5.0,42.6,4.4,12.0,12.5,6.6
2,01001020300,AL,Alabama,Autauga,01001,3216,2485,POINT (-86.4597033 32.4740243),14.7,80.6,37.7,31.5,13.9,8.5,8.5,85.5,17.6,67.4,7.3,10.7,16.1,57.7,26.3,13.9,19.1,13.1,18.6,15.1,23.7,10.6,8.6,42.9,78.4,41.8,14.1,10.3,25.9,35.1,75.5,17.6,40.3,29.9,4.6,39.0,4.1,9.7,9.0,5.8
3,01001020400,AL,Alabama,Autauga,01001,4246,3344,POINT (-86.4448353 32.4710304),9.3,81.7,33.9,32.5,14.2,7.6,10.3,88.0,14.2,71.5,7.5,9.6,12.9,61.1,24.9,12.8,12.5,7.9,15.8,13.5,19.5,7.9,9.3,42.4,80.1,43.4,9.3,8.3,21.5,31.0,78.1,15.5,36.9,25.5,3.6,35.8,3.8,6.8,6.1,4.7
4,01001020501,AL,Alabama,Autauga,01001,4322,3369,POINT (-86.4225578 32.4478607),9.9,81.3,27.5,27.7,15.6,5.2,9.0,89.0,12.0,70.5,5.4,9.3,10.0,66.3,24.0,10.2,9.5,5.4,14.8,11.0,14.9,6.2,6.8,37.2,77.8,40.5,8.1,6.3,21.1,30.8,79.8,11.2,34.7,20.3,2.6,35.1,2.8,5.5,5.1,3.3


In [17]:
# Create the output folder
output_folder = Path(
    "../Data"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

# Save the final dataset
csv_path = (
    output_folder /
    "places_wide_clean.csv"
)

df_wide.to_csv(
    csv_path,
    index=False
)

print("Dataset saved to:", csv_path)


Dataset saved to: ../Data/places_wide_clean.csv


In [18]:
# Descriptions for reference columns
column_descriptions = {
    "LocationName": "Census tract identifier/name.",
    "StateAbbr": "Two-letter state abbreviation.",
    "StateDesc": "State name.",
    "CountyName": "County name.",
    "CountyFIPS": "Five-digit state and county FIPS code.",
    "TotalPopulation": "Total census tract population.",
    "TotalPop18plus": "Population aged 18 years or older.",
    "Geolocation": "Census tract geographic point location."
}

# Create the data dictionary
dictionary_lines = [
    "# Data Dictionary",
    "",
    "| Column | Description |",
    "|---|---|"
]

for column in df_wide.columns:
    description = column_descriptions.get(
        column,
        f"Crude prevalence percentage for {column}."
    )

    dictionary_lines.append(
        f"| {column} | {description} |"
    )

dictionary_path = (
    output_folder /
    "data_dictionary.md"
)

dictionary_path.write_text(
    "\n".join(dictionary_lines),
    encoding="utf-8"
)

print("Data dictionary saved to:", dictionary_path)


Data dictionary saved to: ../Data/data_dictionary.md


**Final Output**

The final wide dataset was validated and saved as `places_wide_clean.csv`. A `data_dictionary.md` file was also created to explain every remaining column.
